In [14]:
import pickle 
import seaborn as sns
import pandas as pd
import shap
import torch
import torch.nn as nn
import numpy as np
from operator import add
from functools import reduce
from torchvision import models, transforms
from skimage import io
from torch.utils.data import Subset
from model import MultiInputModel
from torch.utils.data import Dataset, DataLoader
import torch
import pandas as pd
import random
import copy
from typing import List
from skimage import io
from params import ROOT_DIR
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold


In [26]:
class OldMultiInputModel(nn.Module):
    """docstring for MultiInputModel."""

    def __init__(self, features_img, input_tab, output_tab, in_features, n_classes):
        super(OldMultiInputModel, self).__init__()
        self.features_img = features_img
        self.tab_mlp = nn.Sequential(
            nn.Linear(input_tab, output_tab), nn.ReLU(inplace=True)
        )
        self.concat_mlp = nn.Linear(in_features + output_tab, n_classes)

    def forward(self, img, tab):
        output_img = self.features_img(img)
        output_tab = self.tab_mlp(tab)

        output_img_tab = torch.cat((output_img.squeeze(), output_tab), dim=1)

        output = self.concat_mlp(output_img_tab)

        return output

In [7]:
single_model =  models.regnet_y_800mf(False)
num_ftrs = single_model.fc.in_features
single_model.fc = nn.Linear(num_ftrs, 2)


In [8]:
single_model.load_state_dict(torch.load("../models/bk_models/regnet_single_intput_1.pth"))

<All keys matched successfully>

In [20]:
regnet_multi = models.regnet_y_800mf(False)
features = nn.Sequential(*list(regnet_multi.children()))[:-1]
in_features = 784
ft_size = 8
output_tab = 3
n_classes = 2
features_2 = copy.deepcopy(features)
multi_model_double = MultiInputModel(
    features, features_2, ft_size, output_tab, in_features, n_classes
)


In [21]:
multi_model_double.load_state_dict(torch.load("../models/regnet_multi_input_double_img_1k_fold.pth"))

<All keys matched successfully>

In [27]:
regnet_multi = models.regnet_y_800mf(False)
features = nn.Sequential(*list(regnet_multi.children()))[:-1]
in_features = 784
ft_size = 8
output_tab = 3
n_classes = 2

multi_model = OldMultiInputModel(
    features, ft_size, output_tab, in_features, n_classes
        )

In [28]:
multi_model.load_state_dict(torch.load("../models/bk_models/regnet_multi_input_1.pth"))

<All keys matched successfully>

In [29]:
print(multi_model)

OldMultiInputModel(
  (features_img): Sequential(
    (0): SimpleStemIN(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (1): Sequential(
      (block1): AnyStage(
        (block1-0): ResBottleneckBlock(
          (proj): ConvNormActivation(
            (0): Conv2d(32, 64, kernel_size=(1, 1), stride=(2, 2), bias=False)
            (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
          (f): BottleneckTransform(
            (a): ConvNormActivation(
              (0): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
              (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): ReLU(inplace=True)
            )
            (b): ConvNormActivation(
              (0): Conv2d(64, 64, kernel_size=(3, 3), str

In [30]:
print(multi_model_double)

MultiInputModel(
  (features_img_1): Sequential(
    (0): SimpleStemIN(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (1): Sequential(
      (block1): AnyStage(
        (block1-0): ResBottleneckBlock(
          (proj): ConvNormActivation(
            (0): Conv2d(32, 64, kernel_size=(1, 1), stride=(2, 2), bias=False)
            (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
          (f): BottleneckTransform(
            (a): ConvNormActivation(
              (0): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
              (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): ReLU(inplace=True)
            )
            (b): ConvNormActivation(
              (0): Conv2d(64, 64, kernel_size=(3, 3), stri

In [35]:
for param in multi_model_double.features_img_1.parameters():
    print(param.requires_graduires_grad)

AttributeError: 'Parameter' object has no attribute 'requires_graduires_grad'

In [69]:
sum_weights = 0
for child in multi_model_double.features_img_1.children():
    for name, param in child.named_parameters():
        sum_weights += torch.sum(torch.sum(param[0].data))
        

In [70]:
sum_weights

tensor(13.3010)

In [71]:
sum_weights = 0
for child in multi_model_double.features_img_2.children():
    for name, param in child.named_parameters():
        sum_weights += torch.sum(torch.sum(param[0].data))
        

In [72]:
sum_weights

tensor(12.8785)

In [73]:
sum_weights = 0
for child in multi_model_double.tab_mlp.children():
    for name, param in child.named_parameters():
        sum_weights += torch.sum(torch.sum(param[0].data))
        

In [74]:
sum_weights

tensor(-1.0706)